In [ ]:
import pandas as pd
import numpy as np

# CARGA DEL DATASET DESDE ARCHIVO
json = "../data/raw/streaming_users_dirty.json"
df_raw = pd.read_json(json)

print("=== Dataset original cargado ===")

df_raw.head()

=== Dataset original cargado ===


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1


In [6]:
# Vista de Dimensiones
df_raw.shape

(8160, 8)

El dataset contiene 8160 filas en 8 columnas.

In [7]:
# Tipos de Datos
print("Tipos de datos: ")
df_raw.info()

Tipos de datos: 
<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 510.1 KB


### Diccionario

| Variable | Descripción |
| :--- | :--- |
| `user_id` | Identificador único de usuario |
| `age` | Edad del usuario |
| `subscription_plan` | Plan contratado |
| `monthly_watch_time_mins` | Minutos vistos mensuales |
| `country` | País de residencia |
| `favorite_genre` | Género cinematográfico favorito |
| `last_login_date` | Última fecha de inicio de sesión |
| `customer_support_tickets` | Reclamos realizados |

Se observan nulos en las columnas monthly_watch_time_mins, favorite_genre y last_login_date. Además, la columna last_login_date tiene el formato texto (str) cuando debería ser del tipo fecha (datetime).

In [8]:
# Estadisticas para columnas numericas
df_raw.describe().round(2)

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.00,8160.00,7967.00,8160.00
mean,13995.43,34.10,1107.35,1.80
std,2310.81,14.51,5310.44,11.33
min,10000.00,-5.00,-120.00,-1.00
25%,11987.75,25.00,489.20,0.00
50%,13998.50,33.00,757.40,1.00
75%,15997.25,42.00,1045.70,1.00
max,17999.00,150.00,99999.00,150.00


Se observan valores negativos en las columnas age, monthly_watch_time_mins y customer_support_tickets, los cuales son imposibles en el contexto del dataset. Además, se detectaron outliers extremos: un valor de 150 años para la edad, un tiempo de reproducción de 99.999 minutos (el cual supera los minutos totales que tiene un mes) y un número excesivo de tickets de soporte (150).

In [9]:
# Suma de cantidad de nulos por columnas
df_raw.isnull().sum()

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64

In [10]:
# Porcentaje de faltantes por variable
faltantes = df_raw.isnull().sum()
porcentaje = (faltantes / len(df_raw) * 100).round(2)

print("=== Faltantes iniciales ===")
print(pd.DataFrame({'Faltantes': faltantes,
                            '%': porcentaje}).query('Faltantes > 0'))

=== Faltantes iniciales ===
                         Faltantes     %
monthly_watch_time_mins        193  2.37
favorite_genre                 240  2.94
last_login_date                320  3.92


Se observa la presencia de valores nulos en tres columnas del dataset: last_login_date con 320 (3.92%) registros, favorite_genre con 240 (2.94%) y monthly_watch_time_mins con 193 (2.37%).

In [11]:
# Revision de filas duplicadas
columnas_a_revisar = df_raw.columns.difference(['age'])

total_duplicados = df_raw.duplicated(subset=columnas_a_revisar).sum()
print(f"Cantidad de filas duplicadas (sin contar 'edad'): {total_duplicados}")

Cantidad de filas duplicadas (sin contar 'edad'): 128


Se identificaron 128 filas duplicadas excluyendo la columna age. Se excluyó esta columna para verificar si existen registros idénticos de un mismo usuario pero con edades distintas asignadas, lo cual representaría un error de consistencia en los perfiles.

In [12]:
# Revision de la columna last login date
# 1. Convertimos a fecha temporalmente y extraemos el año
anios = pd.to_datetime(df_raw['last_login_date'], errors='coerce').dt.year

# 2. Contamos cuántas filas hay por cada año y ordenamos cronológicamente
filas_por_anio = anios.value_counts().sort_index()

# 3. Calculamos totales generales y los inválidos
total_filas_dataset = len(df_raw)
total_registros_validos = filas_por_anio.sum()
total_nulos_invalidos = total_filas_dataset - total_registros_validos

print("="*55)
print("      DISTRIBUCIÓN DE FILAS POR AÑO Y PORCENTAJE")
print("="*55)
print(f"{'Año':<15}     | {'Registros':<15} | {'Porcentaje':<12}")
print("-"*55)

# Imprimimos los años válidos
for anio, cantidad in filas_por_anio.items():
    porcentaje = (cantidad / total_filas_dataset) * 100
    print(f"• Año {int(anio):<9}     | {cantidad:<15} | {porcentaje:>6.2f}%")

# 4. Agregamos la fila resumen de nulos/inválidos al final de la tabla
porcentaje_invalidos = (total_nulos_invalidos / total_filas_dataset) * 100
print(f"• Nulos/Inválidos   | {total_nulos_invalidos:<15} | {porcentaje_invalidos:>6.2f}%")

print("="*55)
# Resumen de lo calculado
print(f"TOTAL FILAS DATASET | {total_filas_dataset:<15} | 100.00%")
print("="*55)

      DISTRIBUCIÓN DE FILAS POR AÑO Y PORCENTAJE
Año                 | Registros       | Porcentaje  
-------------------------------------------------------
• Año 2018          | 863             |  10.58%
• Año 2019          | 930             |  11.40%
• Año 2020          | 870             |  10.66%
• Año 2021          | 919             |  11.26%
• Año 2022          | 920             |  11.27%
• Año 2023          | 906             |  11.10%
• Año 2024          | 966             |  11.84%
• Año 2025          | 1002            |  12.28%
• Año 2029          | 15              |   0.18%
• Nulos/Inválidos   | 769             |   9.42%
TOTAL FILAS DATASET | 8160            | 100.00%


Se detectaron 15 registros con una fecha futura (año 2029) que resultan imposibles para el contexto actual. Además, se identificaron 769 filas entre valores nulos e inválidos (un 9.42% del dataset).

In [14]:
#  Revision de valores unicos y cantidades en variables categoricas

print("=== CANTIDAD DE USUARIOS POR PLAN DE SUSCRIPCIÓN ===")
print(df_raw['subscription_plan'].value_counts().sort_index())
print("-" * 50)  # Línea separadora estática

print("\n=== CANTIDAD DE USUARIOS POR PAÍS ===")
print(df_raw['country'].value_counts().sort_index())
print("-" * 50)

print("\n=== CANTIDAD DE USUARIOS POR GÉNERO FAVORITO ===")
print(df_raw['favorite_genre'].value_counts().sort_index())
print("-" * 50)

=== CANTIDAD DE USUARIOS POR PLAN DE SUSCRIPCIÓN ===
subscription_plan
BASICO         52
Basic          52
Básico       3450
Estándar     2711
Estándar       46
PREMIUM        26
Premium      1519
Premium        31
Premiun        23
STANDARD       34
Std            48
basico         60
básico         50
estandar       36
premium        22
Name: count, dtype: int64
--------------------------------------------------

=== CANTIDAD DE USUARIOS POR PAÍS ===
country
ARG             10
Argentina     1087
Argentina       10
BRA             15
Brasil        1132
Brazil          21
CHL             18
COL             19
Chile         1132
Chile           16
Colombia      1116
MEX             16
Mexico          15
México        1129
PER             16
Peru            15
Perú          1120
URY             17
Uruguay       1124
argentina       16
brasil          13
chile           15
colombia        27
méxico          25
perú            12
uruguay         24
Name: count, dtype: int64
---------------

Se detectaron inconsistencias de formato en las columnas subscription_plan, country y favorite_genre, incluyendo textos en mayúsculas, minúsculas, variaciones con y sin tilde, errores de tipeo y abreviaturas.